# ISOM 835 · Session 6 — Regression: Predicting Numbers
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Oct 26 · Prof. Hasan Arslan**

**Tonight in one sentence:** for five weeks our models answered *yes or no*; tonight they answer **how much**: *what is this house worth?*

**How to use this notebook:** run the cells top to bottom (Shift + Enter). Read the short text above each cell first: it tells you **what the cell does** and **how to read the output**. You do not need to understand every line of code tonight. You do need to understand every *result*. (The first code cell only checks your Python setup: run it and move on.)

🎮 **Play along:** the [House Price Explorer](https://isom-835.vercel.app/explore/regression/) turns tonight's four big ideas into small games you can drag on your phone.

### Tonight's map

| Step | The question it answers |
|---|---|
| **1** | How do we grade a prediction that is a number? |
| **2–3** | What does one straight line say? What does *"holding everything else fixed"* mean? |
| **4–5** | What happens with all 79 features, and how can a model *memorize*? |
| **6–7** | How does a *complexity tax* (ridge, lasso) fix that? Why train on log(price)? |
| **8–9** | Where is the model wrong, and which number do we report to whom? |
| **10** | Your turn |

Look for three kinds of boxes: **How to read it** (what an output means), **Stop and notice** (the lesson), and **🧠 Quick check** (a 20-second question to test yourself; click to see the answer).

### Nine words you will hear tonight

| Word | Plain English |
|---|---|
| **Regression** | A model that predicts a **number** (a price, a demand, a delivery time) instead of yes/no. |
| **Feature** | One input column the model uses: size, quality, neighborhood… |
| **Coefficient** (slope) | How much the prediction moves when one feature goes up by one unit, **holding the others fixed**. *"$107 per square foot."* |
| **Residual** | The miss on one house: **actual price − predicted price**. |
| **MAE** | The **average** miss, in dollars. |
| **RMSE** | Also a size of miss in dollars, but **big misses count extra**. |
| **R²** | Share of the price differences the model explains. **0** = no better than guessing the average. **1** = perfect. On new houses it can even go **negative**: worse than guessing. |
| **Overfitting** | Memorizing the training houses instead of learning the pattern. Great on homework, bad on the exam. |
| **Ridge / Lasso** | Linear regression with a **complexity tax** that stops it from overfitting. Lasso can switch features off completely. |

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## The business situation

You are a pricing analyst at a company that **buys homes, fixes them up, and resells them**. Before every offer you need one number: *what is this house really worth?*

- Offer **too much** and the company loses money when it resells.
- Offer **too little** and the seller walks away.

This is not a made-up worry. In 2021 **Zillow** shut down its home-buying business after it bought thousands of homes for more than it could resell them for: write-downs of more than $540 million, and a quarter of its staff cut. Its CEO: *"the unpredictability in forecasting home prices far exceeds what we anticipated."* Tonight we build a simple cousin of the models Zillow used, and we learn how to check where it goes wrong.

**The data:** 1,460 homes sold in **Ames, Iowa** (2006–2010), with **79 facts** about each one.

| Column | What it means |
|---|---|
| `SalePrice` | what the house sold for (**our target**) |
| `GrLivArea` | living area above ground, in square feet |
| `OverallQual` | overall quality of materials and finish, from 1 (very poor) to 10 (excellent) |
| `YearBuilt` | the year the house was built |
| `GarageCars` | how many cars fit in the garage |
| `Neighborhood` | one of 25 neighborhoods in Ames |

The next cell loads the houses and splits them like in every session: **80% to train** the model, **20% held back to test** it, like an exam the model has not seen.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, Lasso
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore', message='Skipping features without any observed values')   # harmless notes when a column
warnings.filterwarnings('ignore', message='Currently, when `keep_empty_feature')               # is empty in a small sample

def to_k(v, _=None):
    # Chart labels in thousands of dollars: 150000 → $150K
    return f'-${-v / 1000:,.0f}K' if v < 0 else f'${v / 1000:,.0f}K'

# Load the houses (the same data as fetch_openml('house_prices') on the Datasets page)
URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/ames_housing.csv'
houses = pd.read_csv(URL)

# Target (y): the sale price in dollars
y = houses['SalePrice']

# Inputs (X): the 79 other columns. Id is just a row number, so we drop it.
X = houses.drop(columns=['Id', 'SalePrice'])

# 80% to train, 20% held back as the exam
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=835)

print(f'{len(houses):,} houses · {X.shape[1]} features · median price ${y.median():,.0f}')
print(f'{len(X_tr):,} houses to train on · {len(X_te):,} houses in the test set\n')
houses[['SalePrice', 'GrLivArea', 'OverallQual', 'YearBuilt', 'GarageCars', 'Neighborhood']].head()

## 1. How do we grade a number?

With yes/no we counted right and wrong. With a price, the model is almost never *exactly* right, so we measure **how far off** it is. Two ways:

- **MAE** (mean absolute error): drop the minus signs and average the misses. *"On average we are off by $X."*
- **RMSE** (root mean squared error): square each miss, average, then take the square root. Squaring makes **big misses count extra**.

A tiny example first: three houses, and we missed by $10K, $10K, and $100K.

In [ ]:
misses = np.array([10_000, 10_000, 100_000])

print(f'MAE  = average of the misses              = ${np.abs(misses).mean():,.0f}')
print(f'RMSE = square, average, then square root  = ${np.sqrt((misses ** 2).mean()):,.0f}   ← the one big miss pulls it up')

**Stop and notice:** same three misses, two different answers. RMSE is bigger because it punishes the one $100K disaster. Which one you report depends on whether big misses hurt your business extra (more on this at the end).

### The baseline: guess the average for every house

Every model must beat the laziest possible guess: *"every house is worth the average price."* The next cell sets up a **leaderboard**. Every model we build tonight goes on it, so we can see the progress at the end.

### 🧠 Quick check

Two models price the same four houses.
- **Model A** misses every house by **$20K**.
- **Model B** gets three houses exactly right and misses the fourth by **$80K**.

Which one has the lower MAE? Which has the lower RMSE?

<details><summary>Click for the answer</summary>

**Same MAE:** both average $20K a house. **But RMSE says B is scarier:** A's RMSE is $20K, B's is $40K, because the one $80K miss is squared. If one big overpayment could hurt your company, RMSE is the number that warns you.
</details>

In [ ]:
leaderboard = []   # every model's test score goes here

def score(name, pred):
    # Grade predictions on the test houses and add a row to the leaderboard
    row = {'model': name,
           'MAE': mean_absolute_error(y_te, pred),
           'RMSE': root_mean_squared_error(y_te, pred),
           'R²': round(r2_score(y_te, pred), 2) + 0}   # + 0 turns -0.0 into 0.0
    leaderboard.append(row)
    print(f"{name:34s} MAE ${row['MAE']:>7,.0f}   RMSE ${row['RMSE']:>7,.0f}   R² {row['R²']:5.2f}")

# The baseline: predict the average training price for every test house
average_price = y_tr.mean()
print(f'Baseline guess for every house: ${average_price:,.0f}\n')
score('Baseline: guess the average', np.full(len(y_te), average_price))

**How to read it:** guessing the average misses a house by about **$55K** on average. R² is about **0**, which is what "no better than the average" means. That is the bar to beat.

## 2. One feature, one line

Start with what every buyer asks first: *how big is it?* **Linear regression** draws the straight line through the houses that makes the misses as small as possible. (Precisely: it makes the sum of the **squared** misses smallest, which is why it is also called **least squares**.)

The next cell fits the line and reads it in plain English.

In [ ]:
one = LinearRegression()
one.fit(X_tr[['GrLivArea']], y_tr)   # double brackets keep it a table with one column

intercept = one.intercept_
slope = one.coef_[0]
print(f'price ≈ ${intercept:,.0f} + ${slope:,.0f} × square feet\n')
print(f'In words: each extra square foot adds about ${slope:,.0f} to the price.')
print(f'Example: a 2,000 sq ft house → ${intercept:,.0f} + ${slope:,.0f} × 2,000 = ${intercept + slope * 2000:,.0f}\n')

score('1 feature: living area', one.predict(X_te[['GrLivArea']]))

**How to read the chart below:**
- each **blue dot** is a training house;
- the **green line** is the model;
- each **black dot** is a **new house** from the test set, and its **red line** is a **residual**: how far its real price is from what the line predicted.

Linear regression chose the green line by making the misses on the **training houses** (the blue dots), squared and added up, as small as possible. The red lines show the same kind of miss on houses the model has never seen.

🎮 Try to beat it by hand: [Station 1 of the House Price Explorer](https://isom-835.vercel.app/explore/regression/#s1) lets you drag the line and race least squares.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(X_tr['GrLivArea'], y_tr, s=8, alpha=0.3, color='#5b6bd8', label='training houses')

line_x = pd.DataFrame({'GrLivArea': np.linspace(400, 4600, 50)})
ax.plot(line_x['GrLivArea'], one.predict(line_x), color='#2e8b57', lw=2.5, label='the fitted line')

# Twelve test houses: the black dot is the real price, the red line is the miss
some = X_te.sample(12, random_state=835)
actual = y_te.loc[some.index]
predicted = one.predict(some[['GrLivArea']])
ax.vlines(some['GrLivArea'], predicted, actual, color='#d64545', lw=1.8, label='residual = the miss')
ax.scatter(some['GrLivArea'], actual, s=28, color='black', zorder=3, label='test houses')

ax.set_xlabel('living area (sq ft)'); ax.set_ylabel('sale price')
ax.yaxis.set_major_formatter(to_k)
ax.set_title('Price vs. size: one straight line')
ax.legend(loc='upper left'); plt.show()

**Stop and notice:**
- Size alone gets R² ≈ **0.54**: about **half** of why prices differ is explained by size.
- The average miss drops from about $55K to about **$36K**. Real progress, one column.
- Look at the bottom right: two **huge houses that sold cheap**. Both were new homes sold *before they were finished*. Real data is messy; this kind of sale comes back in Step 8.

## 3. Two features: "holding everything else fixed"

Now add **quality** (`OverallQual`, 1 to 10). Watch what happens to the price of a square foot.

In [ ]:
two = LinearRegression()
two.fit(X_tr[['GrLivArea', 'OverallQual']], y_tr)

per_sqft, per_quality_point = two.coef_
print(f'price ≈ ${per_sqft:,.0f} × square feet + ${per_quality_point:,.0f} × quality point − ${-two.intercept_:,.0f}\n')
print(f'One feature:  a square foot is worth ${slope:,.0f}')
print(f'Two features: a square foot is worth ${per_sqft:,.0f}   ← it dropped by half!')
print(f'How much do size and quality move together? correlation = {X_tr["GrLivArea"].corr(X_tr["OverallQual"]):.2f}\n')

score('2 features: area + quality', two.predict(X_te[['GrLivArea', 'OverallQual']]))

**Why did the square foot get cheaper?** Bigger houses also tend to be **better built** (correlation about 0.6). With only size in the model, the $107 was paying for **both** the extra space **and** the better quality that usually comes with it.

(Don't worry that the equation subtracts about $107K: no real house has 0 square feet and quality 0, so that starting point is never used on its own.)

With quality in the model, **$55** is the value of one extra square foot **for two houses of the same quality**. That is what **"holding everything else fixed"** means, and it is how you read every coefficient from now on.

> ⚠️ **A coefficient is not a cause.** It describes houses that *sold*. Bigger houses also tend to sit on bigger lots, in nicer neighborhoods, with newer kitchens. So "$55 per square foot" is a pattern in past sales, not a promise that building an addition pays back $55 a foot. Say *"houses with more space sell for more"*, not *"adding space adds value."*

## 4. All 79 features: the Session 3 pipeline

Real pricing models use everything: basement, garage, lot, neighborhood, kitchen quality… We reuse the **Session 3 recipe**, the prep station that gets the data ready before the model sees it:

| Column type | Step 1 | Step 2 |
|---|---|---|
| **Number** columns (36) | fill blanks with the median | put them on the same scale (`StandardScaler`) |
| **Text** columns (43) | fill blanks with the word "missing" | one 0/1 column per category (`OneHotEncoder`); categories with fewer than 10 houses are lumped together |

We wrap the recipe in a small function, `prep()`, so that every model gets its **own fresh copy**.

### 🧠 Quick check

In the two-feature model, a square foot is worth **$55**. Which sentence is right?

- **A.** Adding one square foot to any house adds $55 to its price.
- **B.** Between two houses of the **same quality**, each extra square foot goes with about $55 more in price.

<details><summary>Click for the answer</summary>

**B.** A coefficient describes houses that sold, comparing houses that are alike in the other features in the model. It is not a promise about what a renovation will earn (that would be A, a claim about cause).
</details>

In [ ]:
def prep():
    # The Session 3 recipe: picks the number and text columns automatically
    numbers = Pipeline([('fill', SimpleImputer(strategy='median')),
                        ('scale', StandardScaler())])
    text = Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing')),
                     ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=10))])
    # make_column_selector finds the columns by type, so the recipe still works if you drop or add columns
    return ColumnTransformer([('num', numbers, make_column_selector(dtype_include='number')),
                              ('text', text, make_column_selector(dtype_exclude='number'))],
                             verbose_feature_names_out=False)

# Prep station + linear regression, trained on all the training houses
all_linear = Pipeline([('prep', prep()), ('model', LinearRegression())])
all_linear.fit(X_tr, y_tr)

n_columns = len(all_linear.named_steps['prep'].get_feature_names_out())
print(f'79 features become {n_columns} columns after one-hot encoding.\n')

linear_pred = all_linear.predict(X_te)
score('All features: linear regression', linear_pred)

**Stop and notice:** from a $36K average miss (one feature) to about **$18K**. More information, better prices. (Its RMSE is about **$31K**; remember that number, it comes back in Step 5.)

But the model now has **255 knobs** (one coefficient per column) and only **1,168 houses** to learn from. That is a lot of knobs. What happens when there are **more knobs than houses**?

## 5. Overfitting: memorizing vs. learning

Picture a student who gets the answer key to 100 practice questions and **memorizes** it. Perfect score on the practice test. On the real exam, with new questions, lost.

The next cell gives linear regression only **100 houses**, and about 150 columns to play with (fewer than 255, because 100 houses show fewer categories). With more knobs than houses, it can fit every training house **perfectly**. Then we test it on the houses it has never seen.

**All the numbers in Step 5 are RMSE.** That is why guessing the average shows up as about **$74K** here (its RMSE), not the $55K MAE from Step 1.

In [ ]:
# Pretend we only have 100 houses to learn from
small_X = X_tr.sample(100, random_state=835)
small_y = y_tr.loc[small_X.index]

memorizer = Pipeline([('prep', prep()), ('model', LinearRegression())])
memorizer.fit(small_X, small_y)

knobs = len(memorizer.named_steps['prep'].get_feature_names_out())
print(f'100 houses, {knobs} knobs: more knobs than houses.\n')
def grade(label, actual, pred):
    print(f"{label:38s} RMSE ${root_mean_squared_error(actual, pred):>9,.0f}   R² {round(r2_score(actual, pred), 2) + 0:6.2f}")

grade('Practice test (its own 100 houses):', small_y, memorizer.predict(small_X))
grade('Real exam (292 new houses):', y_te, memorizer.predict(X_te))
grade('Just guessing the average:', y_te, np.full(len(y_te), average_price))

**Stop and notice:** a **perfect** score on the houses it trained on (zero error, R² = 1), and a test error **far worse than just guessing the average**. That is why R² on the exam is **negative**. That is overfitting. The model learned the noise of 100 particular houses, not the pattern of house prices.

### The cure: ridge regression, a complexity tax

**Ridge** is linear regression with one extra rule: every coefficient pays a **tax** that grows with its size. A coefficient only gets big if it **earns its keep across many houses**. Wild, memorizing coefficients become too expensive.

- The **tax rate** is called **alpha**. Tiny alpha = plain linear regression. Huge alpha = every coefficient squashed toward zero.
- `RidgeCV` **tries many tax rates** and picks the best one with cross-validation (Session 3), using **only the training houses**. The test set stays a fair exam.

Same 100 houses, now with ridge:

In [ ]:
tax_rates = np.logspace(-3, 3, 50)    # 50 alphas to try, from 0.001 to 1,000

ridge_small = Pipeline([('prep', prep()), ('model', RidgeCV(alphas=tax_rates))])
ridge_small.fit(small_X, small_y)

results = {
    'Guess the average': root_mean_squared_error(y_te, np.full(len(y_te), average_price)),
    'Linear (memorizer)': root_mean_squared_error(y_te, memorizer.predict(X_te)),
    'Ridge (taxed)': root_mean_squared_error(y_te, ridge_small.predict(X_te)),
}
print(f"tax rate chosen by cross-validation: alpha = {ridge_small.named_steps['model'].alpha_:.1f}\n")
for name, err in results.items():
    print(f'{name:20s} test RMSE ${err:>9,.0f}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(list(results), list(results.values()), color=['#999999', '#d64545', '#2e8b57'])
ax.invert_yaxis()   # list the bars top to bottom, in our order
ax.set_xlabel('test RMSE (lower is better)')
ax.xaxis.set_major_formatter(to_k)
ax.set_title('Trained on only 100 houses'); plt.show()

**Stop and notice:** same 100 houses, same columns. The only change is the tax. Ridge gives up a perfect practice score and in return does **many times better** on the real exam.

Even more striking: ridge on just **100** houses (RMSE about $30K) is about as good as plain linear regression on all **1,168** houses (Step 4, RMSE about $31K). A good tax is worth a lot of data.

🎮 [Station 2 of the explorer](https://isom-835.vercel.app/explore/regression/#s2) lets you slide the tax rate yourself and watch practice and exam errors form a **U**.

## 6. Ridge and lasso on all the houses, plus the log trick

Back to all 1,168 training houses. Three upgrades, one at a time. Watch the leaderboard.

### 6a. Ridge on all the houses

In [ ]:
ridge = Pipeline([('prep', prep()), ('model', RidgeCV(alphas=tax_rates))])
ridge.fit(X_tr, y_tr)

score('Ridge', ridge.predict(X_te))
print(f"\ncompare: plain linear regression in Step 4 had RMSE ${root_mean_squared_error(y_te, linear_pred):,.0f}")

### 6b. The log trick: think in percentages, not dollars

Look at the prices (left chart below): most houses sell for $100K–$250K, but a few sell for $500K+. A **$20K miss** means very different things:

- on a $100K house it is a **20%** mistake;
- on a $500K house it is only **4%**.

A model trained on dollars treats both misses the same, so the few expensive houses pull it around. The fix: train on **log(price)** instead (right chart). On the log scale, equal steps are equal **percentages**: look at the labels, every step **doubles** the price. And the shape becomes a nice symmetric hill.

**The recipe:** train on `np.log(y)`, then turn predictions back into dollars with `np.exp(...)` **before** you grade them. (You may also see `np.log1p` and `np.expm1` in other notebooks and in HW3; for prices they do the same job.)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(y_tr, bins=40, color='#5b6bd8')
ax[0].set_title('Sale price: a long tail to the right'); ax[0].set_xlabel('price')
ax[0].xaxis.set_major_formatter(to_k)
ax[1].hist(np.log(y_tr), bins=40, color='#2ea89a')
ax[1].set_title('log(price): a symmetric hill'); ax[1].set_xlabel('log(price), labeled in dollars')
ticks = [50_000, 100_000, 200_000, 400_000, 800_000]   # each step doubles the price
ax[1].set_xticks(np.log(ticks), [to_k(t) for t in ticks])
plt.tight_layout(); plt.show()

ridge_log = Pipeline([('prep', prep()), ('model', RidgeCV(alphas=tax_rates))])
ridge_log.fit(X_tr, np.log(y_tr))               # 1. train on log(price)
ridge_log_pred = np.exp(ridge_log.predict(X_te))  # 2. back to dollars before grading

score('Ridge, log price', ridge_log_pred)

### 6c. Lasso: a tax that switches features off

**Lasso** uses a slightly different tax. Ridge shrinks every coefficient a bit; lasso pushes the useless ones to **exactly zero**. Those features are switched off: lasso does **feature selection for free**.

> **Ridge shrinks. Lasso selects.** (**Elastic net** mixes the two taxes; it is one of the options in HW3.)

`LassoCV` picks its tax rate with cross-validation, just like `RidgeCV`. (It takes a few seconds.)

In [ ]:
lasso = Pipeline([('prep', prep()), ('model', LassoCV(cv=5, random_state=835, max_iter=20_000))])
lasso.fit(X_tr, np.log(y_tr))
lasso_pred = np.exp(lasso.predict(X_te))

score('Lasso, log price', lasso_pred)

coefs = lasso.named_steps['model'].coef_
print(f"\ntax rate chosen by cross-validation: alpha = {lasso.named_steps['model'].alpha_:.4f}")
print(f'Lasso kept {(coefs != 0).sum()} of {len(coefs)} columns and switched off the other {(coefs == 0).sum()}.')

### 🧠 Quick check

You fit lasso and see that **139 of 255** coefficients are exactly **0**. What does that mean in plain English?

<details><summary>Click for the answer</summary>

Lasso decided those 139 columns are not worth their tax, so it **switched them off**: the model ignores them completely. That is feature selection for free. (Ridge would have kept all 255, each turned down a little.)
</details>

**Stop and notice:** average miss about **$14K**, at least as good as ridge on log price ($15K), while lasso uses **fewer than half** of the 255 columns. The tax made the model **simpler**, with no loss in accuracy. (A $0.4K gap on 292 test houses is small; call it a tie, not a win.)

## 7. Read the model: a realtor's checklist

Lasso still keeps over 100 columns, too many to explain to a client. But alpha is a **dial**: turn up the tax and lasso keeps fewer features. How much accuracy do we give up?

(Cross-validation chose alpha ≈ 0.0007 above. Lasso's alphas are on a different scale from ridge's, so don't compare the two numbers. And we peek at the test set here **only to show the trade-off**; in real work you choose alpha with cross-validation on the training houses.)

In [ ]:
for alpha in [0.001, 0.005, 0.01, 0.05]:
    dial = Pipeline([('prep', prep()), ('model', Lasso(alpha=alpha, max_iter=50_000))])
    dial.fit(X_tr, np.log(y_tr))
    kept = (dial.named_steps['model'].coef_ != 0).sum()
    err = root_mean_squared_error(y_te, np.exp(dial.predict(X_te)))
    print(f'alpha {alpha:<6}  keeps {kept:3d} columns   test RMSE ${err:,.0f}')

🎮 [Station 3 of the explorer](https://isom-835.vercel.app/explore/regression/#s3) is this dial, with every surviving feature on screen.

**A business trade-off, not a math one:** at alpha = 0.01 lasso keeps **21 columns** and loses only a few thousand dollars of accuracy. A model you can **explain on one page** is often worth that.

The next cell lists those survivors. How to read the table:
- **price effect** = the **percent change in predicted price** when that feature is one step higher, **holding everything else fixed**.
- **one step** = for a number column, a *typical difference* between houses (one standard deviation, because we scaled the numbers); for a category, *having* that category, compared with houses that don't.

In [ ]:
strict = Pipeline([('prep', prep()), ('model', Lasso(alpha=0.01, max_iter=50_000))])
strict.fit(X_tr, np.log(y_tr))
score('Strict lasso (≈20 features), log', np.exp(strict.predict(X_te)))

names = strict.named_steps['prep'].get_feature_names_out()
coefs = pd.Series(strict.named_steps['model'].coef_, index=names)
kept = coefs[coefs != 0].sort_values(ascending=False)

def one_step(col):
    if col not in X_tr.columns:          # a one-hot column such as MSZoning_RL
        return 'has this category'
    sd = X_tr[col].std()
    return f'+{sd:,.1f}' if sd < 10 else f'+{sd:,.0f}'

meaning = {'OverallQual': 'quality rating (1–10)', 'GrLivArea': 'living area, sq ft', 'YearBuilt': 'year built (newer)',
           'GarageCars': 'garage size, cars', 'OverallCond': 'condition rating (1–10)', 'Fireplaces': 'fireplaces',
           'YearRemodAdd': 'year of last remodel', 'BsmtFullBath': 'full baths in basement', 'TotRmsAbvGrd': 'rooms above ground',
           'TotalBsmtSF': 'basement, sq ft', 'MSZoning_RL': 'zoned residential, low density', 'LotArea': 'lot size, sq ft',
           'FullBath': 'full bathrooms', 'BsmtFinSF1': 'finished basement, sq ft', 'ScreenPorch': 'screen porch, sq ft',
           'WoodDeckSF': 'wood deck, sq ft', '1stFlrSF': 'first floor, sq ft', 'GarageArea': 'garage, sq ft',
           'KitchenAbvGr': 'kitchens', 'PoolArea': 'pool, sq ft', 'MSSubClass': 'building-type code (really a category: ignore)'}

checklist = pd.DataFrame({
    'what it is': [meaning.get(c, '') for c in kept.index],
    'price effect': (np.exp(kept) - 1).map('{:+.1%}'.format),   # log coefficient → percent change
    'one step': [one_step(c) for c in kept.index],
})
checklist

**Read the top of the list out loud:** two houses alike in every other way, but one is **about 1.4 points higher on quality**: it sells for roughly **+13%** more. **About 530 more square feet**: roughly **+10%**. **About 30 years newer**: roughly **+7%**. **0.8 more garage spaces**: roughly **+6%**.

**Read the bottom with care.** The effects get small, and a few look odd (a pool *lowers* the price?). Features that overlap, like garage spaces and garage area, or living area, first floor and rooms, split the credit between them, so one at a time they mean little. Lasso also shrinks every effect a bit, so these are on the cautious side. Trust the top of the list; treat the bottom as noise.

That reads like a realtor's checklist, and that is the point: **a regularized linear model is the most explainable strong model you will ever ship.** (And still: describe, don't claim causes.)

## 8. Residuals: what the model is missing

A metric is one number. A **residual plot** shows *every* miss. For each test house: across = what the model predicted, up = **actual − predicted**.

**How to read a residual plot:**

| You see… | It means… |
|---|---|
| a **random cloud** of the same width everywhere, centered on 0 | ✅ healthy |
| a **fan** (misses grow as prices grow) | errors are proportional to price → use log(price) |
| a **curve** or a frown | a pattern the straight line can't bend to |
| a few dots **far from the rest** | houses the model doesn't understand: look at them! |

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, (name, pred) in zip(ax, [('Linear regression, dollars', linear_pred), ('Lasso, log price', lasso_pred)]):
    a.scatter(pred, y_te - pred, s=12, alpha=0.6, color='#5b6bd8')
    a.axhline(0, color='#2e8b57', lw=2)
    a.set_title(name); a.set_xlabel('predicted price')
    a.xaxis.set_major_formatter(to_k)
ax[0].set_ylabel('residual = actual − predicted')

# Point at the single worst miss of linear regression
worst = np.argmin(y_te.values - linear_pred)
ax[0].annotate(f'priced at {to_k(linear_pred[worst])}\nsold for {to_k(y_te.values[worst])}',
               xy=(linear_pred[worst], y_te.values[worst] - linear_pred[worst]),
               xytext=(-170, 25), textcoords='offset points', color='#d64545',
               arrowprops=dict(arrowstyle='->', color='#d64545'))
ax[0].yaxis.set_major_formatter(to_k)
plt.tight_layout(); plt.show()

**What to look for:**
- **Left (linear regression on dollars):** one dot sits far below the rest. The model priced that house at about **$570K**; it sold for about **$275K**. If we had bought it at our price, we would have **overpaid by almost $300K on one house**. That is the Zillow nightmare in a single dot. We'll find out why in a moment.
- **Right (lasso on log price):** a tighter cloud. The misses still spread out a little more for pricier houses, and **one dot sits high above the rest**: keep an eye on it.

The next cell lists the five biggest misses of our lasso model. Is there something these houses have in common?

In [ ]:
misses = pd.DataFrame({
    'actual': y_te,
    'predicted': lasso_pred.round(-3).astype(int),
    'miss': (y_te - lasso_pred).round(-3).astype(int),
    'sq ft': X_te['GrLivArea'],
    'quality': X_te['OverallQual'],
    'pool sq ft': X_te['PoolArea'],
    'sale condition': X_te['SaleCondition'],
})
misses.reindex(misses['miss'].abs().sort_values(ascending=False).index).head(5)

**How to read `sale condition`:** *Normal* is an ordinary sale. *Partial* means a **new house sold before it was finished**. *Alloca* means two linked properties sold together.

**Stop and notice:** the top row is the **same house** from the left panel. Linear regression *over*priced it by almost $300K; lasso *under*priced it by about $120K. One house fooled both models, in opposite directions. Look at its `pool sq ft`: **738**.

Only **5 of our 1,168 training houses have a pool**, and this pool is bigger than all of them. With five examples, each model's "value of a pool" is close to a guess: linear regression guessed pools are worth a fortune, lasso guessed they hurt. Then each multiplied its guess by a pool bigger than any it had seen. The next cell asks the obvious question: **what if this house had no pool?**

In [ ]:
pool_house = misses['miss'].abs().idxmax()            # lasso's biggest miss: the house with the pool
print(f"Training houses with a pool: {(X_tr['PoolArea'] > 0).sum()} of {len(X_tr):,}\n")

as_is = X_te.loc[[pool_house]]
no_pool = as_is.copy()
no_pool['PoolArea'] = 0          # erase the pool...
no_pool['PoolQC'] = np.nan       # ...and its quality rating

print(f"Actual sale price:                    ${y_te[pool_house]:,.0f}")
print(f"Linear regression, with the pool:     ${all_linear.predict(as_is)[0]:,.0f}")
print(f"Linear regression, if it had no pool: ${all_linear.predict(no_pool)[0]:,.0f}")
print(f"Lasso, with the pool:                 ${np.exp(lasso.predict(as_is))[0]:,.0f}")
print(f"Lasso, if it had no pool:             ${np.exp(lasso.predict(no_pool))[0]:,.0f}")

**Stop and notice:**
- Without the pool, the two models finally **agree** (about $190K). But both still miss by about **$85K**. The pool explains the wild *disagreement*; the rest comes from the sale itself: two linked properties sold together (`Alloca`), something the model has almost never seen.
- **A feature that almost no house has gets a wild coefficient**: that's overfitting again, hiding in a single column. When one house fools every model, **look at the house** before you trust the price.
- Three of the five biggest misses are **Partial** sales: new, high-quality homes sold before they were finished. Sale condition *is* one of our columns, but the training data has only about a hundred such sales, so the model knows them poorly. That is a **segment** the model doesn't understand (HW3 asks you to find yours).
- The expensive end is where the model is least sure, and that is exactly where a pricing mistake costs the most. This is the Zillow lesson: **a price without an error bar is a bet.**

## 9. The leaderboard, and which number to report to whom

In [ ]:
board = pd.DataFrame(leaderboard).set_index('model')

colors = ['#999999' if m.startswith('Baseline') else '#2e8b57' if m == board['MAE'].idxmin() else '#5b6bd8' for m in board.index]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(board.index, board['MAE'], color=colors)
for row, mae in enumerate(board['MAE']):
    ax.text(mae + 500, row, to_k(mae), va='center')
ax.invert_yaxis(); ax.set_xlabel('average miss on a test house (MAE)')
ax.xaxis.set_major_formatter(to_k)
ax.set_title('Tonight, from the average to lasso'); plt.show()

print(f'Lasso, log price: on average {mean_absolute_percentage_error(y_te, lasso_pred):.1%} off (MAPE)\n')
board.style.format({'MAE': '${:,.0f}', 'RMSE': '${:,.0f}', 'R²': '{:.2f}'})

### The metric menu: same model, different audiences

| Metric | Reads as | Report it to |
|---|---|---|
| **MAE** | the average miss, in dollars | operations, sales, a home seller: *"on average we're off by about $14K"* |
| **RMSE** | a miss size where big misses count extra | risk and finance: one $150K overpayment hurts more than ten $15K ones |
| **MAPE** | the average miss as a **percent** | executives. ⚠️ **Breaks when actual values can be near zero**: a $5 miss on a $1 item is a 500% error |
| **R²** | share of price differences explained | analysts, comparing models on the same data |

## 10. Your turn

Each question has a starter cell below. Fill in the line marked `# ← your turn`, run it, then answer in words. Work with a neighbor if you like.

1. **Location, location, location.** Fit `LassoCV` on log price **without** the `Neighborhood` column. Hint: `X_tr.drop(columns=['Neighborhood'])` and the same for `X_te`; `prep()` adapts by itself.
   - How much does the test MAE change, in dollars?
   - Surprised by the size of the change? Which other columns might already "know" the neighborhood? One sentence.
2. **Turn the dial.** Make your own strict lasso with `alpha=0.02` and then `alpha=0.03` (Step 7 already showed 0.005 and 0.01). For each: how many columns survive, and what is the test RMSE? Which one would you show a client, and why?
3. **Metric memo.** Write three sentences, no code words: which metric you would report to a **home seller**, to a **mortgage lender**, and to your **data-science lead**, and why.
4. **Challenge (optional).** Put `HistGradientBoostingRegressor(random_state=835)` (from `sklearn.ensemble`) at the end of the same pipeline, trained on log price. Does it beat lasso on this data? Why might a simple model hold its own on only 1,168 houses?

**Q1 · Location, location.** One line to finish: drop `Neighborhood` from the test houses too.

In [ ]:
# Q1 · lasso on log price, without the Neighborhood column
X_tr_noN = X_tr.drop(columns=['Neighborhood'])
X_te_noN = ...   # ← your turn: the same drop, for the test houses (X_te)

no_hood = Pipeline([('prep', prep()), ('model', LassoCV(cv=5, random_state=835, max_iter=20_000))])
no_hood.fit(X_tr_noN, np.log(y_tr))
no_hood_pred = np.exp(no_hood.predict(X_te_noN))

print(f'With Neighborhood:    MAE ${mean_absolute_error(y_te, lasso_pred):,.0f}')
print(f'Without Neighborhood: MAE ${mean_absolute_error(y_te, no_hood_pred):,.0f}')

Your answer to Q1 (double-click to edit): *the MAE changed by about $\_\_\_. I think it changed so little because…*

**Q2 · Turn the dial.** Same loop as Step 7, with two new tax rates. Run it, then decide which model you would show a client.

In [ ]:
# Q2 · how many columns survive, and how accurate is each model?
for alpha in [0.02, 0.03]:   # ← your turn: these are the two tax rates; try others too
    dial = Pipeline([('prep', prep()), ('model', Lasso(alpha=alpha, max_iter=50_000))])
    dial.fit(X_tr, np.log(y_tr))
    kept = (dial.named_steps['model'].coef_ != 0).sum()
    err = root_mean_squared_error(y_te, np.exp(dial.predict(X_te)))
    print(f'alpha {alpha:<6}  keeps {kept:3d} columns   test RMSE ${err:,.0f}')

Your answer to Q2: *I would show the client alpha = \_\_\_ because…*

**Q3 · Metric memo** (double-click to write; no code words):

- *To a home seller I would report… because…*
- *To a mortgage lender I would report… because…*
- *To my data-science lead I would report… because…*

**Q4 · Challenge (optional).** Gradient boosting at the end of the same pipeline. Just run it and compare with lasso.

In [ ]:
# Q4 · does a more flexible model beat lasso here?
from sklearn.ensemble import HistGradientBoostingRegressor

boost = Pipeline([('prep', prep()), ('model', HistGradientBoostingRegressor(random_state=835))])
boost.fit(X_tr, np.log(y_tr))
boost_pred = np.exp(boost.predict(X_te))

print(f'Lasso, log price:             MAE ${mean_absolute_error(y_te, lasso_pred):,.0f}')
print(f'Gradient boosting, log price: MAE ${mean_absolute_error(y_te, boost_pred):,.0f}')
# Why might a simple model hold its own on only 1,168 houses? One sentence below.

## What we learned tonight

- **Regression predicts a number.** Grade it by how far off it is: **MAE** (average miss) and **RMSE** (big misses count extra), against the **guess-the-average** baseline.
- **A coefficient is a slope in business units, holding everything else fixed.** Not a cause.
- **More knobs than data = memorizing.** Ridge and lasso add a **complexity tax**; cross-validation picks the tax rate. **Ridge shrinks, lasso selects.**
- **Skewed target? Train on log(price)**, then `np.exp` back to dollars before you grade.
- **Look at the residuals.** They show where the model is wrong before any single number does.

**HW3** (due **Mon Nov 16, 11:59 PM**, one notebook): Part 1 is the threshold work on your HW2 credit model from Session 5; **Part 2 is tonight**: an Ames pipeline with ridge, lasso, or elastic net on log price, MAE and RMSE in dollars against the baseline, a residual plot, and one paragraph on the houses your model gets most wrong. **Competition:** team formed and one submission by **Mon Nov 2**; the leaderboard closes **Mon Nov 9, 4:30 PM**. **Project Milestone 1** (your problem and dataset, one paragraph): **Mon Nov 2**.

🎮 **Before you go:** price a house yourself in [Station 4 of the explorer](https://isom-835.vercel.app/explore/regression/#s4): an honest range, a receipt, and a warning when the house leaves the data.

**Next week:** decision trees and random forests, the first models that need no scaling at all.